# Recs 005: Structured game embeddings (QA)

## Key Goal

Validate structured-preference embedding artifacts produced by the pipeline job.

## Decision It Supports

Whether the structured embedding job outputs are complete, aligned, and ready for downstream eval/retrieval.

## Primary Metrics

Artifact integrity and parity checks (coverage, dimensions, app-id alignment, metadata sanity).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")


REPO_ROOT = _repo_root()
EMBED_DEFAULT_DIR = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_profile" / "default"
EMBED_STRUCTURED_DIR = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_profile" / "structured_eval"
REVIEWS_LONG_PATH = EMBED_DEFAULT_DIR / "game_profile_reviews.parquet"
OUT_NPZ = EMBED_STRUCTURED_DIR / "game_profile_embeddings_structured_eval.npz"
OUT_INDEX = EMBED_STRUCTURED_DIR / "game_profile_embedding_index_structured_eval.parquet"
OUT_META = EMBED_STRUCTURED_DIR / "game_profile_embedding_meta_structured_eval.json"

print("Repo root:", REPO_ROOT)
print("Input reviews:", REVIEWS_LONG_PATH)
print("Structured artifacts:")
print("-", OUT_NPZ)
print("-", OUT_INDEX)
print("-", OUT_META)


Repo root: /home/ryanr/workspace/steam_recommendations
Input reviews: /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_reviews.parquet
Structured artifacts:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_embeddings_structured_eval.npz
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_embedding_index_structured_eval.parquet
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_embedding_meta_structured_eval.json


In [ ]:
if not REVIEWS_LONG_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {REVIEWS_LONG_PATH}. Run scripts/recs_job_game_profiles.py first."
    )

for path in [OUT_NPZ, OUT_INDEX, OUT_META]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run scripts/recs_job_game_embeddings.py with structured config first."
        )

reviews_long = pd.read_parquet(REVIEWS_LONG_PATH)
reviews_long = reviews_long.dropna(subset=["review"]).copy()
reviews_game_count = int(reviews_long["app_id"].nunique())

with np.load(OUT_NPZ) as z:
    X_struct = np.asarray(z["embeddings"], dtype=np.float32)
    app_ids_struct = np.asarray(z["app_id"], dtype=np.int64)

idx_struct = pd.read_parquet(OUT_INDEX)
meta_struct = json.loads(OUT_META.read_text(encoding="utf-8"))

print("Reviews rows:", len(reviews_long))
print("Reviews games:", reviews_game_count)
print("Structured matrix shape:", X_struct.shape)
print("Structured app_ids:", len(app_ids_struct))
print("Structured index rows:", len(idx_struct))
print("Structured meta n_games:", meta_struct.get("n_games"), "dim:", meta_struct.get("dim"))


Reviews rows: 62147
Reviews games: 315
Structured matrix shape: (315, 512)
Structured app_ids: 315
Structured index rows: 315
Structured meta n_games: 315 dim: 512


In [ ]:
# Integrity checks (QA-only)
if X_struct.ndim != 2:
    raise RuntimeError(f"Expected 2D embeddings matrix, got shape={X_struct.shape}")

if len(app_ids_struct) != X_struct.shape[0]:
    raise RuntimeError(
        f"NPZ mismatch: app_id len={len(app_ids_struct)} != matrix rows={X_struct.shape[0]}"
    )

if len(idx_struct) != X_struct.shape[0]:
    raise RuntimeError(
        f"Index mismatch: index rows={len(idx_struct)} != matrix rows={X_struct.shape[0]}"
    )

if not np.array_equal(idx_struct["app_id"].to_numpy(dtype=np.int64), app_ids_struct):
    raise RuntimeError("App ID order mismatch between NPZ and index parquet.")

if int(meta_struct.get("n_games", -1)) != int(X_struct.shape[0]):
    raise RuntimeError(
        f"Meta mismatch for n_games: meta={meta_struct.get('n_games')} rows={X_struct.shape[0]}"
    )

if int(meta_struct.get("dim", -1)) != int(X_struct.shape[1]):
    raise RuntimeError(
        f"Meta mismatch for dim: meta={meta_struct.get('dim')} dim={X_struct.shape[1]}"
    )

if meta_struct.get("text_mode") != "structured":
    raise RuntimeError(
        f"Expected structured text_mode in meta, got: {meta_struct.get('text_mode')!r}"
    )

print("Structured artifact QA checks passed.")
print("n_games:", X_struct.shape[0], "dim:", X_struct.shape[1])
print("meta method:", meta_struct.get("method"))
print("meta model:", meta_struct.get("model_name"))


Structured artifact QA checks passed.
n_games: 315 dim: 512
meta method: encode_each_structured_review_mean_pool_per_app_id_l2_normalize
meta model: https://tfhub.dev/google/universal-sentence-encoder/4


In [ ]:
# Optional reference check: compare structured artifact game coverage against input reviews.
print("coverage parity check (reviews games vs structured games):", reviews_game_count, X_struct.shape[0])
if reviews_game_count != X_struct.shape[0]:
    print("Warning: coverage differs. Check whether upstream profile artifact changed filtering/capping logic.")

coverage parity check (reviews games vs structured games): 315 315


## Standard QA gate

Standardized artifact QA checks (same structure as `recs_002`):

- shape/alignment checks
- app-id uniqueness and set parity
- embedding norm sanity (L2)
- metadata completeness
- compact fingerprint output

In [ ]:
required_meta_keys = {
    "model_name",
    "backend",
    "method",
    "text_mode",
    "max_chars_per_review",
    "n_games",
    "dim",
    "n_review_rows_encoded",
}

checks = []

# Alignment and shape checks
checks.append(("matrix_is_2d", bool(X_struct.ndim == 2), f"shape={X_struct.shape}"))
checks.append(("npz_app_id_len_matches_rows", bool(len(app_ids_struct) == X_struct.shape[0]), f"app_ids={len(app_ids_struct)} rows={X_struct.shape[0]}"))
checks.append(("index_rows_match_rows", bool(len(idx_struct) == X_struct.shape[0]), f"index_rows={len(idx_struct)} rows={X_struct.shape[0]}"))

# Uniqueness + order + set parity
idx_app = idx_struct["app_id"].to_numpy(dtype=np.int64)
checks.append(("index_app_id_unique", bool(pd.Series(idx_app).is_unique), f"unique={pd.Series(idx_app).is_unique}"))
checks.append(("npz_index_order_match", bool(np.array_equal(idx_app, app_ids_struct)), "order_equal"))
checks.append(("id_set_parity_vs_reviews", bool(set(idx_app.tolist()) == set(reviews_long["app_id"].astype(int).unique().tolist())), f"idx={len(set(idx_app.tolist()))} reviews={reviews_long['app_id'].nunique()}"))

# Norm sanity
norms = np.linalg.norm(X_struct, axis=1)
checks.append(("norms_finite", bool(np.isfinite(norms).all()), "all finite"))
checks.append(("norms_near_one", bool(np.allclose(norms, 1.0, atol=1e-3)), f"min={norms.min():.6f} mean={norms.mean():.6f} max={norms.max():.6f}"))

# Metadata sanity
meta_keys_ok = required_meta_keys.issubset(set(meta_struct.keys()))
checks.append(("meta_required_keys_present", bool(meta_keys_ok), f"missing={sorted(required_meta_keys.difference(set(meta_struct.keys())))}"))
checks.append(("meta_n_games_match", bool(int(meta_struct.get("n_games", -1)) == int(X_struct.shape[0])), f"meta={meta_struct.get('n_games')} rows={X_struct.shape[0]}"))
checks.append(("meta_dim_match", bool(int(meta_struct.get("dim", -1)) == int(X_struct.shape[1])), f"meta={meta_struct.get('dim')} dim={X_struct.shape[1]}"))
checks.append(("meta_text_mode_structured", bool(meta_struct.get("text_mode") == "structured"), f"text_mode={meta_struct.get('text_mode')}"))

qa_df = pd.DataFrame(checks, columns=["check", "pass", "detail"])
display(qa_df)

if not qa_df["pass"].all():
    failed = qa_df.loc[~qa_df["pass"], ["check", "detail"]]
    raise RuntimeError("Structured artifact QA gate failed:\n" + failed.to_string(index=False))

print("Structured QA gate passed.")
print("Fingerprint app_ids[:5]:", app_ids_struct[:5].tolist())
print("Fingerprint vec[0,:5]:", X_struct[0, :5].tolist())

,check,pass,detail
0,matrix_is_2d,True,"shape=(315, 512)"
1,npz_app_id_len_matches_rows,True,app_ids=315 rows=315
2,index_rows_match_rows,True,index_rows=315 rows=315
3,index_app_id_unique,True,unique=True
4,npz_index_order_match,True,order_equal
5,id_set_parity_vs_reviews,True,idx=315 reviews=315
6,norms_finite,True,all finite
7,norms_near_one,True,min=1.000000 mean=1.000000 max=1.000000
8,meta_required_keys_present,True,missing=[]
9,meta_n_games_match,True,meta=315 rows=315


Structured QA gate passed.
Fingerprint app_ids[:5]: [70, 240, 420, 620, 2870]
Fingerprint vec[0,:5]: [-0.037294451147317886, -0.08125617355108261, -0.05077918991446495, -0.03170423582196236, 0.035468608140945435]
